# RetailPulse — Real Retail Analytics with Apache PySpark RDD

**Dataset:** UCI Machine Learning Repository, *Online Retail* (dataset 352), DOI: https://doi.org/10.24432/C5BW33. The source describes historical transactions from a UK online retailer between 1 Dec 2010 and 9 Dec 2011. UCI reports 541,909 rows. Currency is GBP.

**Important:** this is genuine historical data, not live data, not Pakistani market data, and not representative of every retailer. The source workbook is downloaded from UCI during this notebook; it is not replaced by synthetic data.

This notebook downloads the original Excel workbook directly from the official UCI static file when available, then validates all eight original fields. If direct download fails, manually upload `Online Retail.xlsx` from the UCI dataset page.

## 1. Install packages
Run this cell separately. No `findspark` is needed in Google Colab.

In [ ]:
%pip -q install pyspark pandas openpyxl plotly streamlit

## 2. Download the real UCI workbook
Official dataset page: https://archive.ics.uci.edu/dataset/352/online-retail

In [ ]:
import os, re, sys, subprocess
import pandas as pd
import numpy as np
from pathlib import Path

RAW_XLSX = "/content/Online Retail.xlsx"
RAW_CSV = "/content/Online_Retail_Raw.csv"

if not os.path.exists(RAW_XLSX):
    url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
    zip_path = "/content/uci_online_retail.zip"
    try:
        import urllib.request, zipfile
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            candidates = [n for n in z.namelist() if n.lower().endswith(('.xlsx', '.xls'))]
            if not candidates:
                raise FileNotFoundError("No Excel workbook found inside the UCI archive")
            extracted = z.extract(candidates[0], "/content/uci_online_retail")
            os.replace(extracted, RAW_XLSX)
        print("Downloaded original workbook from UCI.")
    except Exception as e:
        print("Automatic download failed:", repr(e))
        print("Upload the original 'Online Retail.xlsx' workbook from the official UCI page.")
        from google.colab import files
        uploaded = files.upload()
        excel_files = [name for name in uploaded if name.lower().endswith(('.xlsx', '.xls'))]
        if not excel_files:
            raise ValueError("Please upload the original UCI Excel workbook (.xlsx or .xls).")
        RAW_XLSX = "/content/" + excel_files[0]

print("Source file:", RAW_XLSX)
raw_pdf = pd.read_excel(RAW_XLSX, engine="openpyxl")
print("Original data shape:", raw_pdf.shape)
print("Original headers:", raw_pdf.columns.tolist())
display(raw_pdf.head())

## 3. Validate and standardize the original columns
This workflow reads the original workbook directly. It does not assemble ID columns from the repository API, avoiding the missing `InvoiceNo` / `StockCode` issue.

In [ ]:
def canonical_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).strip().lower())

rename_map = {
    "invoiceno": "InvoiceNo", "stockcode": "StockCode",
    "description": "Description", "quantity": "Quantity",
    "invoicedate": "InvoiceDate", "unitprice": "UnitPrice",
    "customerid": "CustomerID", "country": "Country"
}
raw_pdf = raw_pdf.rename(columns={c: rename_map.get(canonical_name(c), str(c).strip()) for c in raw_pdf.columns})
raw_pdf = raw_pdf.loc[:, ~raw_pdf.columns.duplicated()].copy()
required = ["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
missing = [c for c in required if c not in raw_pdf.columns]
if missing:
    raise ValueError(f"Wrong/incomplete file. Missing {missing}. Please use the original UCI Online Retail.xlsx workbook.")
raw_pdf = raw_pdf[required].copy()
print("All eight original columns verified.")
print("Rows:", len(raw_pdf))
print("Columns:", raw_pdf.columns.tolist())
raw_pdf.to_csv(RAW_CSV, index=False)
print("Saved untouched raw source export:", RAW_CSV)

## 4. Start Spark and load the raw source

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("RetailPulse-UCI-Online-Retail-RDD")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
print("Spark version:", spark.version)
source_df = (spark.read.option("header", True)
             .option("inferSchema", True)
             .option("timestampFormat", "M/d/yyyy H:mm")
             .csv(RAW_CSV))
print("Spark source row count:", source_df.count())
source_df.printSchema()
source_df.show(5, truncate=False)
raw_rdd = source_df.rdd

## 5. Data quality audit with RDD
Counts below are issue counts, not mutually exclusive row counts: one row can have more than one flag.

In [ ]:
def quality_flags(row):
    d = row.asDict(recursive=True)
    flags = []
    required_fields = ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice"]
    if any(d.get(k) is None for k in required_fields): flags.append("missing_required_field")
    invoice = str(d.get("InvoiceNo") or "").strip()
    try: qty = float(d.get("Quantity"))
    except (TypeError, ValueError): qty = None
    try: price = float(d.get("UnitPrice"))
    except (TypeError, ValueError): price = None
    if invoice.upper().startswith("C"): flags.append("cancellation_invoice")
    if qty is not None and qty < 0: flags.append("negative_quantity_return")
    if qty is not None and qty <= 0: flags.append("non_positive_quantity")
    if price is not None and price <= 0: flags.append("non_positive_unit_price")
    if d.get("CustomerID") is None: flags.append("missing_customer_id")
    return [(f, 1) for f in (flags or ["no_quality_flag"])]

audit = dict(raw_rdd.flatMap(quality_flags).reduceByKey(lambda a,b: a+b).collect())
print("RDD data-quality audit")
for key, value in sorted(audit.items()): print(f"{key}: {value:,}")
print("Exact duplicate raw rows:", raw_rdd.count() - raw_rdd.map(lambda r: tuple(str(x) for x in r)).distinct().count())

## 6. Clean valid positive sale lines with RDD
Rules: require invoice/stock code/quantity/date/unit price; exclude cancellation invoices, returns (negative quantities), zero/negative quantities, and non-positive prices; retain anonymous customer sales for total revenue; calculate line revenue in GBP.

In [ ]:
import datetime

def clean_sale(row):
    d = row.asDict(recursive=True)
    try:
        invoice = str(d["InvoiceNo"]).strip()
        stock = str(d["StockCode"]).strip()
        qty = float(d["Quantity"])
        price = float(d["UnitPrice"])
        dt = pd.to_datetime(d["InvoiceDate"], errors="coerce")
        if pd.isna(dt) or not invoice or not stock: return None
        if invoice.upper().startswith("C") or qty <= 0 or price <= 0: return None
        cust = d.get("CustomerID")
        if cust is None or pd.isna(cust): customer = "Anonymous"
        else:
            try: customer = str(int(float(cust)))
            except (TypeError, ValueError): customer = "Anonymous"
        desc = d.get("Description")
        country = d.get("Country")
        return {
            "InvoiceNo": invoice, "StockCode": stock,
            "Description": str(desc).strip() if desc is not None and not pd.isna(desc) else "Unknown product",
            "Quantity": qty, "InvoiceDate": dt.to_pydatetime(),
            "UnitPrice": price, "CustomerID": customer,
            "Country": str(country).strip() if country is not None and not pd.isna(country) else "Unknown",
            "Revenue": qty * price
        }
    except (KeyError, TypeError, ValueError, OverflowError):
        return None

mapped_rdd = raw_rdd.map(clean_sale)
valid_rdd = mapped_rdd.filter(lambda x: x is not None).cache()
print("Raw source rows:", raw_rdd.count())
print("Valid positive sale lines before exact deduplication:", valid_rdd.count())

## 7. Remove exact duplicate lines only
An invoice may contain multiple product lines. Never deduplicate using `InvoiceNo` alone.

In [ ]:
deduplicated_rdd = (valid_rdd
    .map(lambda r: (tuple(sorted((k, str(v)) for k, v in r.items())), r))
    .reduceByKey(lambda first, second: first)
    .values().cache())
print("Unique valid sale lines:", deduplicated_rdd.count())

## 8. Demonstrate core PySpark RDD operations

In [ ]:
# map
revenue_rdd = deduplicated_rdd.map(lambda r: r["Revenue"])
print("Revenue sample:", revenue_rdd.take(5))

# filter
high_value_rdd = deduplicated_rdd.filter(lambda r: r["Revenue"] >= 100)
print("Sale lines with revenue >= GBP 100:", high_value_rdd.count())

# flatMap + distinct
product_words_rdd = (deduplicated_rdd.flatMap(lambda r: r["Description"].upper().split())
                     .filter(lambda w: len(w) > 2).distinct())
print("Distinct product description words:", product_words_rdd.take(20))

# reduceByKey
country_revenue_rdd = deduplicated_rdd.map(lambda r: (r["Country"], r["Revenue"])).reduceByKey(lambda a,b: a+b)
month_revenue_rdd = deduplicated_rdd.map(lambda r: (r["InvoiceDate"].strftime("%Y-%m"), r["Revenue"])).reduceByKey(lambda a,b: a+b)
print("Top country totals:", sorted(country_revenue_rdd.collect(), key=lambda x: x[1], reverse=True)[:10])
print("Monthly totals:", sorted(month_revenue_rdd.collect()))

## 9. Business KPIs and analysis tables

In [ ]:
clean_pdf = pd.DataFrame(deduplicated_rdd.collect())
clean_pdf["InvoiceDate"] = pd.to_datetime(clean_pdf["InvoiceDate"])
clean_pdf["Revenue"] = pd.to_numeric(clean_pdf["Revenue"])
clean_pdf["Month"] = clean_pdf["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sales = clean_pdf.groupby("Month", as_index=False)["Revenue"].sum().sort_values("Month")
country_sales = clean_pdf.groupby("Country", as_index=False)["Revenue"].sum().sort_values("Revenue", ascending=False)
product_sales = (clean_pdf.groupby(["StockCode", "Description"], as_index=False)
                 .agg(Revenue=("Revenue", "sum"), Units=("Quantity", "sum"))
                 .sort_values("Revenue", ascending=False))
customer_sales = (clean_pdf[clean_pdf["CustomerID"] != "Anonymous"]
                  .groupby("CustomerID", as_index=False)
                  .agg(Revenue=("Revenue", "sum"), Orders=("InvoiceNo", "nunique"), Units=("Quantity", "sum"))
                  .sort_values("Revenue", ascending=False))

kpis = {
    "raw_rows": int(raw_pdf.shape[0]),
    "clean_sale_lines": int(len(clean_pdf)),
    "unique_invoices": int(clean_pdf["InvoiceNo"].nunique()),
    "identified_customers": int(clean_pdf.loc[clean_pdf.CustomerID != "Anonymous", "CustomerID"].nunique()),
    "unique_products": int(clean_pdf["StockCode"].nunique()),
    "countries": int(clean_pdf["Country"].nunique()),
    "sales_revenue_gbp": round(float(clean_pdf["Revenue"].sum()), 2),
    "units_sold": round(float(clean_pdf["Quantity"].sum()), 2)
}
display(pd.DataFrame([kpis]))
display(monthly_sales.head(12))
display(country_sales.head(10))
display(product_sales.head(10))

## 10. Export cleaned data and analysis outputs

In [ ]:
OUTPUT_DIR = "/content/RetailPulse_Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
clean_pdf.to_csv(f"{OUTPUT_DIR}/cleaned_transactions.csv", index=False)
monthly_sales.to_csv(f"{OUTPUT_DIR}/monthly_sales.csv", index=False)
country_sales.to_csv(f"{OUTPUT_DIR}/country_sales.csv", index=False)
product_sales.to_csv(f"{OUTPUT_DIR}/product_sales.csv", index=False)
customer_sales.to_csv(f"{OUTPUT_DIR}/customer_sales.csv", index=False)
pd.DataFrame([kpis]).to_csv(f"{OUTPUT_DIR}/business_kpis.csv", index=False)
print("Exported:", os.listdir(OUTPUT_DIR))

## 11. Visualize monthly revenue

In [ ]:
import plotly.express as px
fig = px.line(monthly_sales, x="Month", y="Revenue", markers=True,
              title="Monthly Retail Sales Revenue (GBP)")
fig.update_layout(yaxis_title="Revenue (£ GBP)", xaxis_title="Invoice month")
fig.show()

## 12. Download outputs

In [ ]:
from google.colab import files
files.download(f"{OUTPUT_DIR}/cleaned_transactions.csv")
files.download(f"{OUTPUT_DIR}/monthly_sales.csv")
files.download(f"{OUTPUT_DIR}/country_sales.csv")
files.download(f"{OUTPUT_DIR}/product_sales.csv")
files.download(f"{OUTPUT_DIR}/customer_sales.csv")
files.download(f"{OUTPUT_DIR}/business_kpis.csv")

## 13. Launch the RetailPulse website from Colab
Upload the provided `app.py` into the Colab Files panel before running this cell. The temporary public tunnel should only be used with public/demo data, never confidential company data.

In [ ]:
%pip -q install streamlit pyngrok
import os, subprocess, time
from pyngrok import ngrok
if not os.path.exists("/content/app.py"):
    raise FileNotFoundError("Upload app.py into the Colab Files panel first.")
public_url = ngrok.connect(8501)
print("RetailPulse website:", public_url)
streamlit_process = subprocess.Popen(["streamlit", "run", "/content/app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
print("Streamlit process started. Open the URL above.")

## Dataset provenance, citation and limitations

Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33. Official page: https://archive.ics.uci.edu/dataset/352/online-retail. License: CC BY 4.0.

This is a real historical dataset from one UK online retailer (2010–2011), with monetary values in GBP. Cancellation and return lines are excluded from positive-sales KPIs; anonymous transactions remain in overall sales but are excluded from customer-level metrics. Spark runs in local mode (`local[*]`) in Colab; this uses actual RDD APIs but is not a multi-node cluster. Validate business rules before operational financial use.